# Bootstrap executável — NestJS CLI Monorepo (apps por domínio)

Este notebook **edita arquivos do seu repositório** quando você executar as células.

Ele garante:
- `HealthController` (`/health`) em cada app
- `AppModule` registra `HealthController`
- `main.ts` usa porta padrão por app (via env `PORT` com fallback)
- validações rápidas (mostra diffs e recomendações)

## Apps alvo (ajuste se precisar)
- users-api: 3001
- llm-ops-api: 3002
- sharepoint-api: 3003
- sync-api: 3004
- monorepo-ai-llm: 3000 (opcional; mantido por padrão)

## Segurança
- As células fazem backup simples criando `*.bak` antes de sobrescrever.
- Você pode inspecionar as mudanças com `git diff`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import shutil
import subprocess
from dataclasses import dataclass

ROOT = Path('.').resolve()

@dataclass(frozen=True)
class AppSpec:
    name: str
    port: int

# Ajuste aqui se você quiser excluir algum app
APPS: list[AppSpec] = [
    AppSpec('users-api', 3001),
    AppSpec('llm-ops-api', 3002),
    AppSpec('sharepoint-api', 3003),
    AppSpec('sync-api', 3004),
    AppSpec('monorepo-ai-llm', 3000),
]

def run(cmd: list[str]) -> str:
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(cmd)}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}")
    return p.stdout.strip()

def backup_file(path: Path) -> None:
    if path.exists():
        bak = path.with_suffix(path.suffix + '.bak')
        shutil.copy2(path, bak)

def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    backup_file(path)
    path.write_text(content, encoding='utf-8')

def read_text(path: Path) -> str:
    return path.read_text(encoding='utf-8')

print('Repo root:', ROOT)
print('Apps configured:', [a.name for a in APPS])

## 1) Pré-checks (estrutura esperada)

Esta célula valida se as pastas `apps/<app>/src` existem.

In [ ]:
missing = []
for app in APPS:
    app_src = ROOT / 'apps' / app.name / 'src'
    if not app_src.exists():
        missing.append(str(app_src))

if missing:
    raise FileNotFoundError('Algumas pastas de app não existem:\n' + '\n'.join(missing))
else:
    print('OK: todas as pastas apps/<app>/src existem')

## 2) Criar/atualizar `HealthController` em cada app

Cria `apps/<app>/src/health.controller.ts` se não existir (ou sobrescreve, com backup `.bak`).

In [ ]:
HEALTH_CONTROLLER_TS = """import { Controller, Get } from '@nestjs/common';

@Controller('health')
export class HealthController {
  @Get()
  getHealth() {
    return { status: 'ok' };
  }
}
"""

for app in APPS:
    path = ROOT / 'apps' / app.name / 'src' / 'health.controller.ts'
    write_text(path, HEALTH_CONTROLLER_TS)
    print('Wrote', path.relative_to(ROOT))

print('\nDone. Use `git diff` para inspecionar alterações.')

## 3) Garantir que `AppModule` registra `HealthController`

Regras:
- adiciona `import { HealthController } from './health.controller';` se faltar
- garante `controllers: [...]` contendo `HealthController`
- garante `export class AppModule {}` existe

Observação: essa célula tenta editar com regex de forma segura para o template padrão do NestJS CLI.
Se seu arquivo estiver muito diferente, ela vai abortar com erro para evitar estragar.


In [ ]:
def ensure_health_in_app_module(app_module_path: Path) -> None:
    s = read_text(app_module_path)
    original = s

    # 1) garantir import do HealthController
    if "./health.controller" not in s:
        # insere após AppService import (template padrão)
        m = re.search(r"import \{ AppService \} from '\./app\.service';\s*\n", s)
        if not m:
            raise ValueError(f"Não consegui encontrar import do AppService em {app_module_path}")
        insert_at = m.end()
        s = s[:insert_at] + "import { HealthController } from './health.controller';\n" + s[insert_at:]

    # 2) garantir HealthController em controllers
    m = re.search(r"controllers:\s*\[(.*?)\]", s, flags=re.DOTALL)
    if not m:
        raise ValueError(f"Não encontrei a propriedade controllers em {app_module_path}")
    inside = m.group(1)
    if "HealthController" not in inside:
        # adiciona no final, respeitando vírgulas
        inside_stripped = inside.strip()
        if inside_stripped == "":
            new_inside = "HealthController"
        else:
            # garante que termina com vírgula/novo item
            new_inside = inside_stripped + ", HealthController"
        s = s[:m.start(1)] + new_inside + s[m.end(1):]

    # 3) garantir export class AppModule
    if re.search(r"export\s+class\s+AppModule\s*\{\s*\}", s) is None:
        # se existir "export class AppModule" sem chaves, tentamos normalizar
        if re.search(r"export\s+class\s+AppModule\b", s):
            # tenta substituir qualquer export class AppModule ... por export class AppModule {}
            s = re.sub(r"export\s+class\s+AppModule\s*\{[^}]*\}\s*", "export class AppModule {}\n", s)
        else:
            # anexa no final
            s = s.rstrip() + "\n\nexport class AppModule {}\n"

    if s != original:
        write_text(app_module_path, s)

for app in APPS:
    p = ROOT / 'apps' / app.name / 'src' / 'app.module.ts'
    if not p.exists():
        raise FileNotFoundError(f"app.module.ts não existe: {p}")
    ensure_health_in_app_module(p)
    print('Patched', p.relative_to(ROOT))

print('\nDone. Rode `git diff` para revisar.')

## 4) Ajustar `main.ts` para porta por app

Regra:
- substituir `await app.listen(...)` por:
  - `const port = process.env.PORT ? Number(process.env.PORT) : <PORT>;`
  - `await app.listen(port);`

Se o arquivo `main.ts` estiver fora do padrão, a célula aborta para evitar alterações incorretas.


In [ ]:
def patch_main_ts(main_path: Path, port: int) -> None:
    s = read_text(main_path)
    original = s

    # encontra await app.listen(<algo>);
    m = re.search(r"await\s+app\.listen\(([^)]*)\);", s)
    if not m:
        raise ValueError(f"Não encontrei 'await app.listen(...)' em {main_path}")

    replacement = f"const port = process.env.PORT ? Number(process.env.PORT) : {port};\n  await app.listen(port);"
    s = re.sub(r"await\s+app\.listen\(([^)]*)\);", replacement, s, count=1)

    if s != original:
        write_text(main_path, s)

for app in APPS:
    main_path = ROOT / 'apps' / app.name / 'src' / 'main.ts'
    patch_main_ts(main_path, app.port)
    print('Patched', main_path.relative_to(ROOT), '-> default port', app.port)

print('\nDone. Teste rodando `npm run start:dev:<app>` e `curl /health`.')

## 5) Validação: checar se `/health` existe e se `AppModule` contém HealthController

Esta célula não altera nada — só valida o estado.


In [ ]:
def validate_app(app: AppSpec) -> list[str]:
    issues = []
    app_root = ROOT / 'apps' / app.name / 'src'
    health = app_root / 'health.controller.ts'
    if not health.exists():
        issues.append('missing health.controller.ts')

    app_module = app_root / 'app.module.ts'
    s = read_text(app_module)
    if './health.controller' not in s:
        issues.append('AppModule missing HealthController import')
    if 'HealthController' not in s:
        issues.append('AppModule missing HealthController registration')
    if re.search(r"export\s+class\s+AppModule\s*\{\s*\}", s) is None:
        issues.append('AppModule missing export class AppModule {}')

    main_ts = app_root / 'main.ts'
    ms = read_text(main_ts)
    if f": {app.port}" not in ms:
        issues.append(f"main.ts may not have default port {app.port}")
    if 'process.env.PORT' not in ms:
        issues.append('main.ts not using PORT env')
    return issues

all_ok = True
for app in APPS:
    issues = validate_app(app)
    if issues:
        all_ok = False
        print(f"❌ {app.name} issues:")
        for i in issues:
            print('  -', i)
    else:
        print(f"✅ {app.name} OK")

if all_ok:
    print('\nTudo OK. Próximo passo recomendado: ajustar scripts do root e preparar Docker/compose minimal por app.')
else:
    print('\nHá issues. Rode as células anteriores novamente ou corrija manualmente.')

## 6) Comandos úteis (manual)

Depois de rodar as células e revisar `git diff`:

```bash
git diff
npm run start:dev:users
curl -sS http://localhost:3001/health

git add .
git commit -m "chore: add health + ports across domain apps"
```
